# Análise de Vendas em E-commerce no Brasil

**Aluno:** Guilherme Oliveira de Moraes  
**Disciplina:** Linguagem de Programação — Análise e Visualização de Dados com Python  
**Professor:** Alexandre Neves Louzada

## Objetivo e problema
Descrever a evolução das vendas e comparar categorias, canais e localidades, além de explorar a relação entre logística e avaliação. A análise usa o CSV fornecido para o trabalho, sem projetar causalidade.

## Base de dados
A base tem **4,440 registros e 16 colunas**, cobrindo **2015-01 a 2024-12**. Cada linha é um registro de venda; não há identificador único de venda ou dimensão de clientes.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
ROOT=Path.cwd()
if not (ROOT/'dados').exists(): ROOT=ROOT.parent
CSV=ROOT/'dados'/'simulacao_ecommerce_brasil.csv'
DB=ROOT/'database'/'ecommerce.db'
df=pd.read_csv(CSV,parse_dates=['data'])
df.head()

## Leitura e inspeção
Verifique dimensões, tipos, valores ausentes, duplicidade e limites. Os indicadores seguintes são recalculados quando as células são executadas.

In [ ]:
print('Dimensões:',df.shape)
display(df.dtypes.to_frame('tipo'))
display(df.isna().sum().to_frame('nulos'))
print('Duplicados:',df.duplicated().sum())
display(df.describe(include='all').T)

## Limpeza e consistência
Conversão explícita de tipos, remoção de duplicatas exatas e checagem de coerência aritmética. Não se imputam observações ausentes: a base atual não apresenta nulos.

In [ ]:
df=df.drop_duplicates().copy()
num=['ano','mes','quantidade','preco_unitario','faturamento','custo','lucro','prazo_entrega','avaliacao_cliente']
for c in num: df[c]=pd.to_numeric(df[c],errors='coerce')
df['data']=pd.to_datetime(df['data'],errors='coerce')
print('Nulos após conversão:',int(df.isna().sum().sum()))
df['lucro_calculado']=df.faturamento-df.custo
df['diferenca_lucro']=(df.lucro-df.lucro_calculado).abs()
print('Maior diferença entre lucro informado e faturamento menos custo:',df.diferenca_lucro.max())
print('Valores fora da faixa de avaliação (0–5):',int((~df.avaliacao_cliente.between(0,5)).sum()))

## Engenharia de atributos
Mês textual, período mensal, margem baseada no lucro informado e faixas de prazo ajudam a comparar os registros.

In [ ]:
nomes={1:'Jan',2:'Fev',3:'Mar',4:'Abr',5:'Mai',6:'Jun',7:'Jul',8:'Ago',9:'Set',10:'Out',11:'Nov',12:'Dez'}
df['mes_nome']=df.mes.map(nomes)
df['periodo']=df.data.dt.to_period('M')
df['margem_lucro']=np.where(df.faturamento.ne(0),df.lucro/df.faturamento,np.nan)
df['faixa_entrega']=pd.cut(df.prazo_entrega,[0,3,5,7,20],labels=['Até 3 dias','4–5 dias','6–7 dias','Acima de 7 dias'])
df[['mes_nome','periodo','margem_lucro','faixa_entrega']].head()

## Análise exploratória e indicadores

In [ ]:
print('Período:',df.data.min().date(),'a',df.data.max().date())
print('Registros:',len(df),'| Unidades:',df.quantidade.sum())
print('Regiões/UFs/cidades:',df.regiao.nunique(),df.uf.nunique(),df.cidade.nunique())
print('Faturamento total:',df.faturamento.sum())
print('Lucro informado total:',df.lucro.sum())
print('Margem agregada (lucro informado/faturamento):',df.lucro.sum()/df.faturamento.sum())
print('Ticket médio por registro:',df.faturamento.mean())
print('Prazo médio:',df.prazo_entrega.mean(),'| Avaliação média:',df.avaliacao_cliente.mean())


## Gráficos e comparações
Os gráficos usam agregações calculadas diretamente da tabela fornecida.

In [ ]:
mensal=df.groupby('periodo').faturamento.sum()
mensal.plot(figsize=(12,4),title='Faturamento mensal'); plt.ylabel('R$'); plt.show()
display(df.groupby('categoria').agg(faturamento=('faturamento','sum'),lucro=('lucro','sum'),unidades=('quantidade','sum')).sort_values('faturamento',ascending=False))
display(df.groupby(['regiao','uf']).faturamento.sum().sort_values(ascending=False).head(15))
display(df.groupby('canal_venda').agg(registros=('data','size'),faturamento=('faturamento','sum'),lucro=('lucro','sum')).sort_values('faturamento',ascending=False))

In [ ]:
pivot=df.pivot_table(index='ano',columns='mes',values='faturamento',aggfunc='sum')
plt.figure(figsize=(12,5)); sns.heatmap(pivot,cmap='Blues'); plt.title('Faturamento por ano e mês'); plt.show()
sns.scatterplot(data=df.sample(min(1800,len(df)),random_state=42),x='faturamento',y='lucro',hue='categoria',alpha=.6); plt.show()


## Logística, satisfação e associação
A correlação é descritiva e não implica causa. Faixas facilitam a comparação das médias.

In [ ]:
display(df.groupby('faixa_entrega',observed=False).agg(registros=('prazo_entrega','size'),prazo_medio=('prazo_entrega','mean'),avaliacao_media=('avaliacao_cliente','mean'),faturamento=('faturamento','sum')))
display(df[['prazo_entrega','avaliacao_cliente','faturamento','lucro','quantidade']].corr(numeric_only=True))
sns.regplot(data=df,x='prazo_entrega',y='avaliacao_cliente',scatter_kws={'alpha':.2}); plt.show()

## Interpretações dos dados fornecidos
No conjunto completo, o faturamento total é **R$ 85.128.670,31** e o lucro informado total é **R$ 26.886.927,79**, com margem agregada de **31.6%**. A região com maior faturamento é **Sudeste**; a categoria líder em faturamento é **Beleza**; o canal líder é **Marketplace**; o estado líder é **RJ**. O pico mensal agregado ocorre em **11/2022**. Prazo médio: **8.51 dias**; avaliação média: **3.77/5**.

Essas descrições se referem aos valores fornecidos. A base não contém um ID de venda, dados de clientes ou desenho causal.

## Integração com SQLite e SQLAlchemy


In [ ]:
engine=create_engine(f"sqlite:///{DB.as_posix()}")
db=pd.read_sql('SELECT * FROM vendas',engine)
print('Registros no SQLite:',len(db))
display(pd.read_sql('SELECT regiao, SUM(faturamento) AS faturamento FROM vendas GROUP BY regiao ORDER BY faturamento DESC',engine))

## Conclusão
O conjunto permite acompanhar tendências temporais e comparar segmentos comerciais e logísticos. Recomenda-se usar os filtros do dashboard para identificar diferenças de recorte, verificar valores atípicos e tratar correlações como sinais para investigação. As conclusões são descritivas, limitadas às variáveis e ao período presentes no CSV.